# 01. 얼굴 찾기 모델을 CPU · GPU · NPU 에서 돌려 보기

1. 이 컴퓨터에 AI 를 돌릴 장치가 무엇이 있는지 본다
2. OpenVINO 모델 파일을 열어 입력·출력 모양을 읽는다
3. 사진 한 장을 모델이 원하는 모양으로 바꾼다(전처리)
4. 추론하고, 나온 숫자를 얼굴 상자로 바꾼다(후처리)
5. 같은 모델을 CPU · GPU · NPU 에서 돌려 속도를 잰다

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

for d in core.available_devices:
    print(f"{d:6s}", core.get_property(d, "FULL_DEVICE_NAME"))

`NPU` 가 목록에 없으면 NPU 드라이버를 확인한다 (윈도우: 장치 관리자 → 신경망 프로세서 / 우분투: `linux-npu-driver`).

## 2. 모델 파일 열기
OpenVINO 모델은 `.xml`(구조) + `.bin`(가중치) 두 파일이다.

In [ ]:
MODEL = MODELS / "face" / "face-detection-retail-0005.xml"
assert MODEL.exists(), f"모델이 없어요: {MODEL}"

model = core.read_model(MODEL)
print("입력:", model.input(0).any_name, model.input(0).partial_shape)
print("출력:", model.output(0).any_name, model.output(0).partial_shape)

- 입력 `[1, 3, 300, 300]` = 사진 1장 · 색 3채널(B, G, R) · 세로 300 · 가로 300
- 출력 `[1, 1, N, 7]` = 후보 N개, 후보마다 숫자 7개:
  `[이미지 번호, 종류, 점수, x1, y1, x2, y2]` — 좌표는 0~1 (사진 크기에 대한 비율)

## 3. 사진 한 장 — 웹캠, 없으면 파일

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 4. 전처리 — 사진을 모델이 원하는 모양으로

In [ ]:
def preprocess(bgr):
    x = cv2.resize(bgr, (300, 300))     # ① 크기: 300 x 300
    x = x.transpose(2, 0, 1)            # ② 순서: (세로, 가로, 채널) → (채널, 세로, 가로)
    x = x[None].astype(np.float32)      # ③ 맨 앞에 '사진 몇 장' 차원을 붙이고 float32 로
    return x

blob = preprocess(frame)
print(frame.shape, "→", blob.shape, blob.dtype)

## 5. 추론 + 후처리 (CPU)

In [ ]:
compiled = core.compile_model(model, "CPU")
out = compiled(blob)[compiled.output(0)]      # [1, 1, N, 7]
dets = out.reshape(-1, 7)
print("후보 수:", len(dets))

def faces(dets, w, h, conf=0.5):
    boxes = []
    for image_id, label, score, x1, y1, x2, y2 in dets:
        if image_id < 0:                      # -1 = 후보 목록 끝
            break
        if score < conf:
            continue
        boxes.append((int(x1 * w), int(y1 * h), int(x2 * w), int(y2 * h), float(score)))
    return boxes

h, w = frame.shape[:2]
found = faces(dets, w, h)
view = frame.copy()
for x1, y1, x2, y2, s in found:
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 3)
    cv2.putText(view, f"{s:.2f}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)
print("찾은 얼굴:", len(found))
show(view)

## 6. CPU · GPU · NPU 속도 비교
- **컴파일**: 모델을 그 장치용으로 바꾸는 시간 (NPU · GPU 는 처음에 오래 걸린다)
- **첫 추론**: 컴파일 직후 한 번
- **평균**: 예열 뒤 100번 돌린 평균 — 실제 속도는 이것으로 본다

In [ ]:
def bench(device, n=100):
    t0 = time.perf_counter()
    c = core.compile_model(model, device)
    compile_s = time.perf_counter() - t0

    req = c.create_infer_request()
    t0 = time.perf_counter()
    req.infer({0: blob})
    first_ms = (time.perf_counter() - t0) * 1000

    for _ in range(5):                        # 예열
        req.infer({0: blob})
    t0 = time.perf_counter()
    for _ in range(n):
        req.infer({0: blob})
    avg_ms = (time.perf_counter() - t0) / n * 1000
    return compile_s, first_ms, avg_ms

rows = []
for dev in core.available_devices:
    try:
        rows.append((dev, *bench(dev)))
    except Exception as ex:
        print(dev, "실패:", ex)

print(f"{'장치':6s}{'컴파일(s)':>10s}{'첫 추론(ms)':>12s}{'평균(ms)':>10s}{'FPS':>8s}")
for dev, cs, fm, am in rows:
    print(f"{dev:6s}{cs:10.2f}{fm:12.1f}{am:10.2f}{1000 / am:8.0f}")

In [ ]:
# 막대로 보기 — 짧을수록 빠르다
best = min(r[3] for r in rows)
for dev, cs, fm, am in rows:
    print(f"{dev:6s} {'█' * max(1, round(am / best * 10))} {am:.2f} ms")

## 해 볼 것
1. `faces(dets, w, h, conf=0.3)` 과 `conf=0.8` 로 바꿔 보자. 무엇이 달라지나?
2. 장치를 `"AUTO"` 로 컴파일하고 `compiled.get_property("EXECUTION_DEVICES")` 로 무엇을 골랐는지 보자.
3. `core.compile_model(model, "NPU", {"PERFORMANCE_HINT": "THROUGHPUT"})` 와 `"LATENCY"` 를 비교해 보자.
4. 맨 위에 `core.set_property({"CACHE_DIR": "ov-cache"})` 를 넣고 두 번 실행해 컴파일 시간을 비교해 보자.